<a href="https://colab.research.google.com/github/skandanyal/ML-Lab-7th-sem/blob/main/mlLab5/mlLab5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

apply the Candidate-Elimination algorithm. Determine the final version space after processing all examples (positive and negative); identify the most specific and most general hypotheses.

In [1]:
import pandas as pd

# Load the dataset
data = pd.read_csv('/content/spam_dataset.csv')
display(data.head())

,Contains_Link,Unknown_Sender,Spelling_Errors,Urgent_Language,Is_Spam
0,Yes,Yes,Low,Yes,Yes
1,Yes,Yes,High,Yes,Yes
2,No,Yes,High,Yes,No
3,Yes,Yes,High,Yes,Yes
4,No,Yes,Low,No,No


In [2]:
# Define the attributes (features) and the target concept
attributes = data.columns[:-1]  # All columns except the last one ('Is_Spam')
target = data.columns[-1]      # The 'Is_Spam' column

# Initialize the most specific hypothesis (S) with the first positive example
# and the most general hypothesis (G) with all '?' (wildcards)

def initialize_hypotheses(attributes):
    # S: All specific values from the first positive example
    # G: All '?' for all attributes
    specific_h = ['0'] * len(attributes)
    general_h = [['?' for _ in attributes]]
    return specific_h, general_h

# Initialize S and G
specific_hypothesis, general_hypothesis = initialize_hypotheses(attributes)

print("Initial Specific Hypothesis:", specific_hypothesis)
print("Initial General Hypothesis:", general_hypothesis)

Initial Specific Hypothesis: ['0', '0', '0', '0']
Initial General Hypothesis: [['?', '?', '?', '?']]


In [3]:
from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
X = data[attributes]
y = data[target]

# Perform a 90-10 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

print("Training set size:", len(X_train))
print("Testing set size:", len(X_test))

Training set size: 90
Testing set size: 10


In [4]:
import numpy as np

def is_consistent(hypothesis, example):
    """Checks if a hypothesis is consistent with an example."""
    for i in range(len(hypothesis)):
        if hypothesis[i] != '?' and hypothesis[i] != example[i]:
            return False
    return True

def generalize_specific_hypothesis(specific_h, example):
    """Generalizes the specific hypothesis based on a positive example."""
    new_specific_h = list(specific_h)
    for i in range(len(new_specific_h)):
        if new_specific_h[i] == '0':  # '0' means it hasn't seen any example for this attribute yet
            new_specific_h[i] = example[i]
        elif new_specific_h[i] != example[i]:
            new_specific_h[i] = '?'
    return new_specific_h

def specialize_general_hypothesis(general_h, specific_h, example):
    """Specializes the general hypotheses based on a negative example."""
    new_general_h = []
    for h in general_h:
        if is_consistent(h, example): # If the general hypothesis is consistent with the negative example, it must be specialized
            for i in range(len(h)):
                if h[i] == '?':
                    # Create specialized hypotheses for each attribute that is '?'
                    for val in X_train.iloc[:, i].unique(): # Use all possible values from the training data
                        if val != example[i]:
                            h_copy = list(h)
                            h_copy[i] = val
                            # Ensure the new specialized hypothesis is more specific than the specific_h
                            # and not consistent with the negative example
                            if is_consistent(h_copy, specific_h) and not is_consistent(h_copy, example):
                                new_general_h.append(h_copy)
                elif h[i] != example[i] and not is_consistent(h, specific_h):
                    # This case handles when h[i] is already a specific value, but still covers the negative example
                    # and is not more specific than specific_h, it should not be included.
                    # This logic needs refinement to correctly find the minimal specializations.
                    pass
        else:
            # If the general hypothesis is not consistent, keep it
            new_general_h.append(h)
    # Filter out duplicates and inconsistent hypotheses (should be handled by is_consistent check earlier)
    unique_general_h = []
    for h_spec in new_general_h:
        if h_spec not in unique_general_h and not is_consistent(h_spec, example) and is_consistent(specific_h, h_spec): # Check if specific_h is consistent with specialized h_spec
            unique_general_h.append(h_spec)
    return unique_general_h

def candidate_elimination(X_train, y_train, attributes):
    specific_h, general_h = initialize_hypotheses(attributes)

    # Convert DataFrame rows to lists for easier processing
    X_train_list = X_train.values.tolist()
    y_train_list = y_train.values.tolist()

    for i, (example, label) in enumerate(zip(X_train_list, y_train_list)):
        print(f"\nProcessing example {i+1}: {example}, Label: {label}")
        if label == 'Yes':  # Positive example
            # Remove hypotheses in G inconsistent with the positive example
            general_h = [h for h in general_h if is_consistent(h, example)]
            # Generalize specific_h if it's inconsistent
            if not is_consistent(specific_h, example):
                specific_h = generalize_specific_hypothesis(specific_h, example)

        elif label == 'No':  # Negative example
            # Remove hypotheses in S inconsistent with the negative example
            # (specific_h should already be consistent with all positive examples,
            # so if it's consistent with a negative example, it's an error in S development)
            # In general, S should never be consistent with a negative example
            if is_consistent(specific_h, example):
                # This indicates an issue, as S should represent positive examples only.
                # For CE, S is grown from positive examples only. If it matches a negative, it implies G is too general.
                pass # S remains unchanged for negative examples, it only updates with positive ones.

            # Specialize hypotheses in G that are consistent with the negative example
            new_general_h_temp = []
            for h in general_h:
                if is_consistent(h, example):
                    # This hypothesis h needs to be specialized
                    for j in range(len(h)):
                        if h[j] == '?': # Can only specialize '?' attributes
                            # Create specializations by replacing '?' with specific values NOT in the example's attribute
                            for val in X_train.iloc[:, j].unique():
                                if val != example[j]:
                                    h_new_spec = list(h)
                                    h_new_spec[j] = val
                                    # The specialized hypothesis must be consistent with specific_h
                                    # and must not be consistent with the current negative example
                                    if is_consistent(specific_h, h_new_spec) and not is_consistent(h_new_spec, example):
                                        new_general_h_temp.append(h_new_spec)
                else:
                    # If h is already not consistent with the negative example, keep it
                    new_general_h_temp.append(h)

            # Filter out redundant and incorrect hypotheses from new_general_h_temp
            filtered_general_h = []
            for h_spec in new_general_h_temp:
                # Ensure hypothesis is not consistent with any negative example (this example included)
                # and is consistent with the current specific hypothesis
                # And not redundant (already in filtered_general_h)
                is_redundant = False
                for existing_h in filtered_general_h:
                    if np.array_equal(h_spec, existing_h):
                        is_redundant = True
                        break
                if not is_redundant and is_consistent(specific_h, h_spec) and not is_consistent(h_spec, example):
                    filtered_general_h.append(h_spec)
            general_h = filtered_general_h

        print("Specific H:", specific_h)
        print("General H:", general_h)

    return specific_h, general_h

# Run the Candidate-Elimination algorithm on the training data
final_specific_h, final_general_h = candidate_elimination(X_train, y_train, attributes)

print("\nFinal Most Specific Hypothesis (S):", final_specific_h)
print("Final Most General Hypotheses (G):", final_general_h)



Processing example 1: ['No', 'No', 'High', 'Yes'], Label: No
Specific H: ['0', '0', '0', '0']
General H: []

Processing example 2: ['No', 'No', 'Low', 'Yes'], Label: No
Specific H: ['0', '0', '0', '0']
General H: []

Processing example 3: ['No', 'No', 'Low', 'Yes'], Label: No
Specific H: ['0', '0', '0', '0']
General H: []

Processing example 4: ['Yes', 'No', 'Low', 'No'], Label: No
Specific H: ['0', '0', '0', '0']
General H: []

Processing example 5: ['No', 'Yes', 'Low', 'No'], Label: No
Specific H: ['0', '0', '0', '0']
General H: []

Processing example 6: ['No', 'Yes', 'Low', 'No'], Label: No
Specific H: ['0', '0', '0', '0']
General H: []

Processing example 7: ['No', 'Yes', 'Low', 'Yes'], Label: No
Specific H: ['0', '0', '0', '0']
General H: []

Processing example 8: ['No', 'Yes', 'High', 'No'], Label: No
Specific H: ['0', '0', '0', '0']
General H: []

Processing example 9: ['Yes', 'No', 'High', 'No'], Label: No
Specific H: ['0', '0', '0', '0']
General H: []

Processing example 10: 

In [6]:
import numpy as np

def is_consistent(hypothesis, example):
    """Checks if a hypothesis is consistent with an example (i.e., example satisfies hypothesis)."""
    for i in range(len(hypothesis)):
        if hypothesis[i] == '?':  # '?' matches any value
            continue
        if hypothesis[i] != example[i]:
            return False
    return True

def is_more_general_than_or_equal(h_general, h_specific):
    """Checks if hypothesis h_general is more general than or equal to hypothesis h_specific."""
    for i in range(len(h_general)):
        if h_specific[i] == '0':  # h_specific is uninitialized for this attribute; h_general is always more general or equal
            continue
        if h_general[i] == '?':  # h_general is maximally general for this attribute
            continue
        if h_general[i] != h_specific[i]:  # h_general has a specific value different from h_specific's specific value
            return False
    return True

def generalize_specific_hypothesis(specific_h, example):
    """Generalizes the specific hypothesis based on a positive example."""
    new_specific_h = list(specific_h)
    for i in range(len(new_specific_h)):
        if new_specific_h[i] == '0':  # '0' means it hasn't seen any example for this attribute yet
            new_specific_h[i] = example[i]
        elif new_specific_h[i] != example[i]:
            new_specific_h[i] = '?'
    return new_specific_h

def candidate_elimination(X_train, y_train, attributes):
    # Initialize S and G.
    specific_h = ['0'] * len(attributes)
    general_h = [['?' for _ in attributes]]

    # Convert DataFrame rows to lists for easier processing
    X_train_list = X_train.values.tolist()
    y_train_list = y_train.values.tolist()

    for i, (example, label) in enumerate(zip(X_train_list, y_train_list)):
        # print(f"\nProcessing example {i+1}: {example}, Label: {label}")

        if label == 'Yes':  # Positive example
            # Remove hypotheses in G inconsistent with the positive example
            general_h = [h for h in general_h if is_consistent(h, example)]

            # Generalize specific_h if it's inconsistent with the current positive example
            if not is_consistent(specific_h, example):
                specific_h = generalize_specific_hypothesis(specific_h, example)

        elif label == 'No':  # Negative example
            # Update G based on negative example
            temp_general_h = []
            for h in general_h:
                if is_consistent(h, example): # This hypothesis 'h' covers the negative example, it must be specialized
                    # Generate minimal specializations for 'h'
                    for j in range(len(h)):
                        if h[j] == '?': # Only specialize attributes that are '?'
                            possible_values = X_train.iloc[:, j].unique() # Get all unique values for this attribute from training data
                            for val in possible_values:
                                # The new specialized value must not match the negative example's value at this attribute
                                if val != example[j]:
                                    h_new_spec = list(h)
                                    h_new_spec[j] = val

                                    # Ensure the new specialization is still more general than or equal to specific_h
                                    # and is not consistent with the current negative example (implicitly handled by val != example[j])
                                    if is_more_general_than_or_equal(h_new_spec, specific_h):
                                        temp_general_h.append(h_new_spec)
                else:
                    # If h is already not consistent with the negative example, keep it
                    temp_general_h.append(h)

            # Remove duplicates and update general_h
            general_h = []
            for h_spec in temp_general_h:
                # Ensure hypothesis is unique and also still consistent with S after filtering
                if h_spec not in general_h and is_more_general_than_or_equal(h_spec, specific_h):
                    general_h.append(h_spec)

        # print("Specific H:", specific_h)
        # print("General H:", general_h)

    # Filter G to remove any hypothesis that is not more general than S
    # This final filter is important if specific_h was updated after G's elements were added
    general_h = [h for h in general_h if is_more_general_than_or_equal(h, specific_h)]
    # Remove duplicates again after final filter
    unique_general_h = []
    for h_spec in general_h:
        if h_spec not in unique_general_h:
            unique_general_h.append(h_spec)
    general_h = unique_general_h

    return specific_h, general_h

# Run the Candidate-Elimination algorithm on the training data
final_specific_h, final_general_h = candidate_elimination(X_train, y_train, attributes)

print("\nFinal Most Specific Hypothesis (S):", final_specific_h)
print("Final Most General Hypotheses (G):", final_general_h)



Final Most Specific Hypothesis (S): ['Yes', 'Yes', '?', '?']
Final Most General Hypotheses (G): [['Yes', 'Yes', '?', '?']]


In [7]:
def predict(example, specific_h, general_h):
    """Predicts the class label for an example based on the learned version space."""
    # If S and G have converged to a single hypothesis, prediction is straightforward
    if is_consistent(specific_h, example):
        return 'Yes'
    else:
        return 'No'

# Convert X_test and y_test to lists for iteration
X_test_list = X_test.values.tolist()
y_test_list = y_test.values.tolist()

predictions = []
for example in X_test_list:
    predictions.append(predict(example, final_specific_h, final_general_h))

# Calculate accuracy
correct_predictions = 0
for i in range(len(predictions)):
    if predictions[i] == y_test_list[i]:
        correct_predictions += 1

accuracy = (correct_predictions / len(y_test_list)) * 100

print("Model Predictions on Test Set:", predictions)
print("Actual Labels on Test Set:", y_test_list)
print(f"Accuracy on Test Set: {accuracy:.2f}%")

Model Predictions on Test Set: ['No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'Yes']
Actual Labels on Test Set: ['No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'Yes']
Accuracy on Test Set: 100.00%
